In [81]:
import io, requests, pandas as pd
import altair as alt
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [119]:
# EDUCATION ATTAINMENT DATASET
BASE   = "https://sdmx.oecd.org/public/rest"
AGENCY = "OECD.CFE.EDS"
FLOW   = "DSD_REG_EDU@DF_ATTAIN"
VER    = "2.0"

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
    "Accept": "application/json,text/csv,*/*;q=0.8",
}

# 1) Get dimension structure
probe = requests.get(
    f"{BASE}/data/{AGENCY},{FLOW},{VER}",
    params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
    headers=headers, timeout=60
)
probe.raise_for_status()
js = probe.json()

series_dims = js["structure"]["dimensions"]["series"]
dim_order = [d["id"] for d in series_dims]
print("Education dataset dimensions:", dim_order)

# Build codes lookup
codes = {
    d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))}
              for v in d.get("values", [])]
    for d in series_dims
}

def pick_code(dim, prefer_ids=(), prefer_name_contains=()):
    """Pick a valid code for a dimension using id or name hints."""
    vals = codes.get(dim, [])
    # Try id preferences
    for pid in prefer_ids:
        for v in vals:
            if v["id"].upper() == pid.upper():
                return v["id"]
    # Try name contains
    for substr in prefer_name_contains:
        for v in vals:
            if substr.lower() in str(v["name"]).lower():
                return v["id"]
    # Fallback: first available (if any)
    return vals[0]["id"] if vals else ""

# 2) Build proper key with ALL 10 dimensions for education dataset
freq = pick_code("FREQ", prefer_ids=("A",))                                    # Annual
terr_level = "TL2+CTRY"                                                       # BOTH TL2 regions AND countries
ref_area = ""                                                                 # All regions
terr_type = pick_code("TERRITORIAL_TYPE", prefer_ids=("_Z",))                 # Not applicable
measure = pick_code("MEASURE", prefer_ids=("NEAC_SHARE_EA",))                 # Education attainment
age = pick_code("AGE", prefer_ids=("Y25T64",))                                # 25-64 years
sex = pick_code("SEX", prefer_ids=("_T", "T"), prefer_name_contains=("total",)) # Total
education_lev = pick_code("EDUCATION_LEV", prefer_ids=("ISCED11_5T8",))       # Tertiary education
stat_op = pick_code("STATISTICAL_OPERATION", prefer_ids=("MEAN",))            # Mean
unit = pick_code("UNIT_MEASURE", prefer_ids=("PT_POP_SEX_AGE",))              # Percentage

# Build complete key with all 10 dimensions
key_parts = [freq, terr_level, ref_area, terr_type, measure, age, sex, education_lev, stat_op, unit]
key = ".".join(key_parts)
print(f"Education key ({len(key_parts)} parts):", key)

# 3) Download data
url = f"{BASE}/data/{AGENCY},{FLOW},{VER}/{key}"
params = {
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels",
    "startPeriod": "2010",
    "endPeriod": "2024",
}

r = requests.get(url, params=params, headers=headers, timeout=60)

try:
    r.raise_for_status()
    education_df = pd.read_csv(io.StringIO(r.text))
    print(f"\n Education data loaded: {education_df.shape}")
    print("\nFirst few rows:")
    print(education_df.head())
    
    # Show what we got
    print("\nDataset contains:")
    for col in education_df.columns:
        if any(keyword in col.lower() for keyword in ['measure', 'education', 'age', 'territorial']):
            unique_vals = education_df[col].dropna().unique()
            print(f"  {col}: {unique_vals[:3]}{'...' if len(unique_vals) > 3 else ''}")
            
except requests.HTTPError as e:
    print(f" Still failed with: {e}")
    print("Available dimension codes:")
    for d in dim_order:
        print(f"  {d}: {[v['id'] for v in codes.get(d, [])][:5]}")

Education dataset dimensions: ['FREQ', 'TERRITORIAL_LEVEL', 'REF_AREA', 'TERRITORIAL_TYPE', 'MEASURE', 'AGE', 'SEX', 'EDUCATION_LEV', 'STATISTICAL_OPERATION', 'UNIT_MEASURE']
Education key (10 parts): A.TL2+CTRY.._Z.NEAC_SHARE_EA.Y25T64._T.ISCED11_5T8.MEAN.PT_POP_SEX_AGE

 Education data loaded: (6771, 36)

First few rows:
  STRUCTURE                             STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_EDU@DF_ATTAIN(2.0)   

                     STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Educational attainment - Regions      I    A                   Annual   
1  Educational attainment - Regions      I    A                   Annual   
2  Educational attainment - Regions      I    A                   Annual   
3  Educational attainment - Region

In [104]:
## CIRCA 2024 DATASET
# Separate the two territorial levels
EDU_country_data = education_df[education_df['Territorial level'] == 'Country']
EDU_regional_data = education_df[education_df['Territorial level'] == 'TL2']
usa_regional = EDU_regional_data[EDU_regional_data['COUNTRY'] == 'USA']
# Make sure TIME_PERIOD is numeric
education_df['TIME_PERIOD'] = pd.to_numeric(education_df['TIME_PERIOD'], errors='coerce')

# Sort by country and year (descending)
sorted_df = education_df.sort_values(['REF_AREA', 'TIME_PERIOD', 'Sex'], ascending=[True, False, False])

# Keep only the most recent entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='first')

circa_2024 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
EDU_country_data = circa_2024[circa_2024['Territorial level'] == 'Country']
EDU_regional_data = circa_2024[circa_2024['Territorial level'] == 'TL2']


## CIRCA 2010 DATASET
# Keep only the OLDEST entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='last')

circa_2010 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
EDU_country_data_2010 = circa_2010[circa_2010['Territorial level'] == 'Country']
EDU_regional_data_2010 = circa_2010[circa_2010['Territorial level'] == 'TL2']

In [102]:
# CALCULATE USA AND UK COUNTRY VALUES AS MEAN OF REGIONAL VALUES
countries_to_add = []

# Get USA regional data
usa_regional = EDU_regional_data[EDU_regional_data['COUNTRY'] == 'USA']
if len(usa_regional) > 0:
    usa_mean = usa_regional['OBS_VALUE'].mean()
    usa_country_row = {
        'STRUCTURE_NAME': usa_regional.iloc[0]['STRUCTURE_NAME'],
        'TERRITORIAL_LEVEL': 'CTRY',
        'Territorial level': 'Country',
        'REF_AREA': 'USA',
        'Reference area': 'United States',
        'MEASURE': usa_regional.iloc[0]['MEASURE'],
        'Measure': usa_regional.iloc[0]['Measure'],
        'Age': usa_regional.iloc[0]['Age'],
        'Sex': usa_regional.iloc[0]['Sex'],
        'TIME_PERIOD': 2022,
        'Time period': '2022',
        'OBS_VALUE': usa_mean,
        'Observation value': usa_mean,
        'COUNTRY': 'USA',
        'Country': 'United States'
    }
    countries_to_add.append(usa_country_row)
    print(f" USA mean calculated: {usa_mean:.2f}% from {len(usa_regional)} regions")
else:
    print(" No USA regional data found")

# Get UK regional data  
uk_regional = EDU_regional_data[EDU_regional_data['COUNTRY'] == 'GBR']
if len(uk_regional) > 0:
    uk_mean = uk_regional['OBS_VALUE'].mean()
    uk_country_row = {
        'STRUCTURE_NAME': uk_regional.iloc[0]['STRUCTURE_NAME'],
        'TERRITORIAL_LEVEL': 'CTRY',
        'Territorial level': 'Country',
        'REF_AREA': 'GBR',
        'Reference area': 'United Kingdom',
        'MEASURE': uk_regional.iloc[0]['MEASURE'],
        'Measure': uk_regional.iloc[0]['Measure'],
        'Age': uk_regional.iloc[0]['Age'],
        'Sex': uk_regional.iloc[0]['Sex'],
        'TIME_PERIOD': 2022,
        'Time period': '2022',
        'OBS_VALUE': uk_mean,
        'Observation value': uk_mean,
        'COUNTRY': 'GBR',
        'Country': 'United Kingdom'
    }
    countries_to_add.append(uk_country_row)
 
else:
    print("No UK regional data found")

# Get Japan regional data  
jap_regional = EDU_regional_data[EDU_regional_data['COUNTRY'] == 'JPN']
if len(jap_regional) > 0:
    jap_mean = jap_regional['OBS_VALUE'].mean()
    jap_country_row = {
        'STRUCTURE_NAME': jap_regional.iloc[0]['STRUCTURE_NAME'],
        'TERRITORIAL_LEVEL': 'CTRY',
        'Territorial level': 'Country',
        'REF_AREA': 'JPN',
        'Reference area': 'Japan',
        'MEASURE': jap_regional.iloc[0]['MEASURE'],
        'Measure': jap_regional.iloc[0]['Measure'],
        'Age': jap_regional.iloc[0]['Age'],
        'Sex': jap_regional.iloc[0]['Sex'],
        'TIME_PERIOD': 2022,
        'Time period': '2022',
        'OBS_VALUE': jap_mean,
        'Observation value': jap_mean,
        'COUNTRY': 'JPN',
        'Country': 'Japan'
    }
    countries_to_add.append(jap_country_row)
 
else:
    print("No Japan regional data found")

# Add both countries to country data if we have any
if countries_to_add:
    EDU_country_data_with_additions = pd.concat([
        EDU_country_data, 
        pd.DataFrame(countries_to_add)
    ], ignore_index=True)
    
    countries_with_additions = EDU_country_data_with_additions[['Reference area', 'OBS_VALUE']].sort_values('OBS_VALUE', ascending=False)
    
    print(countries_with_additions.head(20))
    
else:
    print(" No countries could be added")
    EDU_country_data_with_additions = EDU_country_data
    countries_with_additions = EDU_country_data[['Reference area', 'OBS_VALUE']].sort_values('OBS_VALUE', ascending=False)

 USA mean calculated: 47.01% from 51 regions
    Reference area  OBS_VALUE
5           Canada  63.400000
30           Korea  57.400000
26         Ireland  56.600000
32      Luxembourg  54.700000
44          Russia  53.100000
22  United Kingdom  52.800000
10          Cyprus  51.400000
52  United Kingdom  51.391667
48          Sweden  50.500000
28          Israel  50.100000
53           Japan  49.760000
39          Norway  49.500000
0        Australia  48.700000
31       Lithuania  47.600000
51   United States  47.011765
6      Switzerland  46.500000
27         Iceland  46.300000
50   United States  46.200000
13         Denmark  45.100000
2          Belgium  45.000000


In [103]:
EDU_country_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_countries_complete.csv", index=False)
EDU_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\education_regional_data.csv", index=False)

# Getting Health Data

In [159]:
# Life expectancy

BASE   = "https://sdmx.oecd.org/public/rest"
AGENCY = "OECD.CFE.EDS"
FLOW   = "DSD_REG_HEALTH@DF_HEALTH"
VER    = "2.0"

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
    "Accept": "application/json,text/csv,*/*;q=0.8",
}

# 1) Get dimension structure
probe = requests.get(
    f"{BASE}/data/{AGENCY},{FLOW},{VER}",
    params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
    headers=headers, timeout=60
)
probe.raise_for_status()
js = probe.json()

series_dims = js["structure"]["dimensions"]["series"]
dim_order = [d["id"] for d in series_dims]
print("life expectancy dataset dimensions:", dim_order)

# Build codes lookup
codes = {
    d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))}
              for v in d.get("values", [])]
    for d in series_dims
}

def pick_code(dim, prefer_ids=(), prefer_name_contains=()):
    """Pick a valid code for a dimension using id or name hints."""
    vals = codes.get(dim, [])
    # Try id preferences
    for pid in prefer_ids:
        for v in vals:
            if v["id"].upper() == pid.upper():
                return v["id"]
    # Try name contains
    for substr in prefer_name_contains:
        for v in vals:
            if substr.lower() in str(v["name"]).lower():
                return v["id"]
    # Fallback: first available (if any)
    return vals[0]["id"] if vals else ""

# 2) Build proper key with ALL 10 dimensions for education dataset
# Build proper key for health dataset
freq = pick_code("FREQ", prefer_ids=("A",))
terr_level = "TL2+CTRY"
ref_area = ""  # All regions/countries, or specify as in SDMX code
terr_type = "" # Usually empty for health
measure = pick_code("MEASURE", prefer_ids=("LFEXP",))  # Life expectancy
age = pick_code("AGE", prefer_ids=("Y0",))              # All ages
sex = ""  # Or "F+M+_T" if you want all sexes
unit = "" # Usually empty for health

# Build complete key (omit education_lev and stat_op)
key_parts = [freq, terr_level, ref_area, terr_type, measure, age, sex, unit]
key = ".".join(key_parts)
print(f"Health key ({len(key_parts)} parts):", key)

# 3) Download data
url = f"{BASE}/data/{AGENCY},{FLOW},{VER}/{key}"
params = {
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels",
    "startPeriod": "2010",
    "endPeriod": "2024",
}

r = requests.get(url, params=params, headers=headers, timeout=60)

try:
    r.raise_for_status()
    health_df = pd.read_csv(io.StringIO(r.text))
    print(f"\n Health data loaded: {health_df.shape}")
    print("\nFirst few rows:")
    print(health_df.head())

    # Show what we got
    print("\nDataset contains:")
    for col in health_df.columns:
        if any(keyword in col.lower() for keyword in ['measure', 'health', 'age', 'territorial']):
            unique_vals = health_df[col].dropna().unique()
            print(f"  {col}: {unique_vals[:3]}{'...' if len(unique_vals) > 3 else ''}")
except requests.HTTPError as e:
    print(f" Still failed with: {e}")
    print("Available dimension codes:")
    for d in dim_order:
        print(f"  {d}: {[v['id'] for v in codes.get(d, [])][:5]}")


life expectancy dataset dimensions: ['FREQ', 'TERRITORIAL_LEVEL', 'REF_AREA', 'TERRITORIAL_TYPE', 'MEASURE', 'AGE', 'SEX', 'UNIT_MEASURE']
Health key (8 parts): A.TL2+CTRY...LFEXP.Y0..

 Health data loaded: (20022, 32)

First few rows:
  STRUCTURE                                STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   

                STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Health statistics - Regions      I    A                   Annual   
1  Health statistics - Regions      I    A                   Annual   
2  Health statistics - Regions      I    A                   Annual   
3  Health statistics - Regions      I    A                   Annual   
4  Health statistics - Regions      I    A            

In [121]:
# Check the data structure and identify time-related columns
print("All columns in the dataset:")
print(health_df.columns.tolist())

print("\nLooking for time/date related columns:")
time_cols = [col for col in health_df.columns if any(keyword in col.lower() 
            for keyword in ['time', 'date', 'year', 'period', 'obs_time'])]
print("Time-related columns:", time_cols)

# Check the first few values of potential time columns
for col in time_cols:
    print(f"\n{col} sample values:")
    print(health_df[col].dropna().unique()[:15])

print(f"\nDataset shape: {health_df.shape}")

All columns in the dataset:
['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'TERRITORIAL_TYPE', 'Territorial typology', 'MEASURE', 'Measure', 'AGE', 'Age', 'SEX', 'Sex', 'UNIT_MEASURE', 'Unit of measure', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country', 'OBS_STATUS', 'Observation status', 'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals']

Looking for time/date related columns:
Time-related columns: ['TIME_PERIOD', 'Time period']

TIME_PERIOD sample values:
[2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023
 2024]

Time period sample values:
[]

Dataset shape: (20022, 32)


In [160]:

vars = health_df[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
data_2022 = vars[vars['TIME_PERIOD'] == 2022]
data_2020 = vars[vars['TIME_PERIOD'] == 2020]
data_2022_us = vars[vars['COUNTRY'] == 'USA']
# Separate the two territorial levels
Health_country_data = vars[vars['Territorial level'] == 'Country']
Health_regional_data = vars[vars['Territorial level'] == 'TL2']

# Make sure TIME_PERIOD is numeric
health_df['TIME_PERIOD'] = pd.to_numeric(health_df['TIME_PERIOD'], errors='coerce')

# Sort by country and year (descending)
sorted_df = health_df.sort_values(['REF_AREA', 'TIME_PERIOD', 'Sex'], ascending=[True, False, False])

# Keep only the most recent entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='first')

circa_2024 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
Health_country_data = circa_2024[circa_2024['Territorial level'] == 'Country']
Health_regional_data = circa_2024[circa_2024['Territorial level'] == 'TL2']

# Keep only OLDEST entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='last')

circa_2010 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
Health_country_data_2010 = circa_2010[circa_2010['Territorial level'] == 'Country']
Health_regional_data_2010 = circa_2010[circa_2010['Territorial level'] == 'TL2']


In [117]:
Health_country_data_2010

,STRUCTURE_NAME,TERRITORIAL_LEVEL,Territorial level,REF_AREA,Reference area,MEASURE,Measure,Age,Sex,TIME_PERIOD,Time period,OBS_VALUE,Observation value,COUNTRY,Country
27,Health statistics - Regions,CTRY,Country,ARG,Argentina,LFEXP,Life expectancy,0 years,Total,2010,NaN,75.3,NaN,ARG,Argentina
656,Health statistics - Regions,CTRY,Country,ARG,Argentina,LFEXP,Life expectancy,0 years,Male,2010,NaN,71.8,NaN,ARG,Argentina
1352,Health statistics - Regions,CTRY,Country,ARG,Argentina,LFEXP,Life expectancy,0 years,Female,2010,NaN,78.6,NaN,ARG,Argentina
46,Health statistics - Regions,CTRY,Country,AUS,Australia,LFEXP,Life expectancy,0 years,Total,2010,NaN,81.7,NaN,AUS,Australia
669,Health statistics - Regions,CTRY,Country,AUS,Australia,LFEXP,Life expectancy,0 years,Male,2010,NaN,79.5,NaN,AUS,Australia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
691,Health statistics - Regions,CTRY,Country,USA,United States,LFEXP,Life expectancy,0 years,Male,2010,NaN,76.2,NaN,USA,United States
1340,Health statistics - Regions,CTRY,Country,USA,United States,LFEXP,Life expectancy,0 years,Female,2010,NaN,81.0,NaN,USA,United States
9,Health statistics - Regions,CTRY,Country,ZAF,South Africa,LFEXP,Life expectancy,0 years,Total,2010,NaN,57.7,NaN,ZAF,South Africa
682,Health statistics - Regions,CTRY,Country,ZAF,South Africa,LFEXP,Life expectancy,0 years,Male,2010,NaN,54.9,NaN,ZAF,South Africa


In [12]:
Health_country_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_countries_complete.csv", index=False)
Health_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_regional_data.csv", index=False)

# Under5 - mortality

In [124]:
# 

BASE   = "https://sdmx.oecd.org/public/rest"
AGENCY = "OECD.CFE.EDS"
FLOW   = "DSD_REG_HEALTH@DF_HEALTH"
VER    = "2.0"

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
    "Accept": "application/json,text/csv,*/*;q=0.8",
}

# 1) Get dimension structure
probe = requests.get(
    f"{BASE}/data/{AGENCY},{FLOW},{VER}",
    params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
    headers=headers, timeout=60
)
probe.raise_for_status()
js = probe.json()

series_dims = js["structure"]["dimensions"]["series"]
dim_order = [d["id"] for d in series_dims]
print("life expectancy dataset dimensions:", dim_order)

# Build codes lookup
codes = {
    d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))}
              for v in d.get("values", [])]
    for d in series_dims
}

def pick_code(dim, prefer_ids=(), prefer_name_contains=()):
    """Pick a valid code for a dimension using id or name hints."""
    vals = codes.get(dim, [])
    # Try id preferences
    for pid in prefer_ids:
        for v in vals:
            if v["id"].upper() == pid.upper():
                return v["id"]
    # Try name contains
    for substr in prefer_name_contains:
        for v in vals:
            if substr.lower() in str(v["name"]).lower():
                return v["id"]
    # Fallback: first available (if any)
    return vals[0]["id"] if vals else ""

# 2) Build proper key with ALL 10 dimensions for education dataset
# Build proper key for health dataset
freq = pick_code("FREQ", prefer_ids=("A",))
terr_level = "TL2+CTRY"
ref_area = ""  # All regions/countries, or specify as in SDMX code
terr_type = "" # Usually empty for health
measure = "MORT_INFANT" # Life expectancy
age = "Y_GE15+Y_LT15+Y_LT1+Y0"              # All ages
sex = ""  # Or "F+M+_T" if you want all sexes
unit = "" # Usually empty for health

# Build complete key (omit education_lev and stat_op)
key_parts = [freq, terr_level, ref_area, terr_type, measure, age, sex, unit]
key = ".".join(key_parts)
print(f"Health key ({len(key_parts)} parts):", key)

# 3) Download data
url = f"{BASE}/data/{AGENCY},{FLOW},{VER}/{key}"
params = {
    "dimensionAtObservation": "AllDimensions",
    "format": "csvfilewithlabels",
    "startPeriod": "2010",
    "endPeriod": "2024",
}

r = requests.get(url, params=params, headers=headers, timeout=60)

try:
    r.raise_for_status()
    health_df = pd.read_csv(io.StringIO(r.text))
    print(f"\n Health data loaded: {health_df.shape}")
    print("\nFirst few rows:")
    print(health_df.head())

    # Show what we got
    print("\nDataset contains:")
    for col in health_df.columns:
        if any(keyword in col.lower() for keyword in ['measure', 'health', 'age', 'territorial']):
            unique_vals = health_df[col].dropna().unique()
            print(f"  {col}: {unique_vals[:3]}{'...' if len(unique_vals) > 3 else ''}")
except requests.HTTPError as e:
    print(f" Still failed with: {e}")
    print("Available dimension codes:")
    for d in dim_order:
        print(f"  {d}: {[v['id'] for v in codes.get(d, [])][:5]}")

life expectancy dataset dimensions: ['FREQ', 'TERRITORIAL_LEVEL', 'REF_AREA', 'TERRITORIAL_TYPE', 'MEASURE', 'AGE', 'SEX', 'UNIT_MEASURE']
Health key (8 parts): A.TL2+CTRY...MORT_INFANT.Y_GE15+Y_LT15+Y_LT1+Y0..

 Health data loaded: (18483, 32)

First few rows:
  STRUCTURE                                STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   

                STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Health statistics - Regions      I    A                   Annual   
1  Health statistics - Regions      I    A                   Annual   
2  Health statistics - Regions      I    A                   Annual   
3  Health statistics - Regions      I    A                   Annual   
4  Health statistics - Regio

In [125]:
vars = health_df[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
data_2022 = vars[vars['TIME_PERIOD'] == 2022]
data_2020 = vars[vars['TIME_PERIOD'] == 2020]
data_2022_us = vars[vars['COUNTRY'] == 'USA']
# Separate the two territorial levels
Health_country_data = vars[vars['Territorial level'] == 'Country']
Health_regional_data = vars[vars['Territorial level'] == 'TL2']

# Make sure TIME_PERIOD is numeric
health_df['TIME_PERIOD'] = pd.to_numeric(health_df['TIME_PERIOD'], errors='coerce')

# Sort by country and year (descending)
sorted_df = health_df.sort_values(['REF_AREA', 'TIME_PERIOD', 'Sex'], ascending=[True, False, False])

# Keep only the most recent entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='first')

circa_2024 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
Health_mortality_country_data = circa_2024[circa_2024['Territorial level'] == 'Country']
Health_mortality_regional_data = circa_2024[circa_2024['Territorial level'] == 'TL2']

# Keep only the OLDEST entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='last')

circa_2010 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
Health_mortality_country_data_2010 = circa_2010[circa_2010['Territorial level'] == 'Country']
Health_mortality_regional_data_2010 = circa_2010[circa_2010['Territorial level'] == 'TL2']

In [15]:
Health_country_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_countries_complete.csv", index=False)
Health_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\health_childmort_regional_data.csv", index=False)

## GDP

In [16]:
gdp_df = pd.read_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\GDP.csv")

In [40]:
import io
import math
import requests
import pandas as pd
import time, random
from typing import Dict, Iterable, List, Mapping, Optional, Tuple, Union
from email.utils import parsedate_to_datetime


BASE = "https://sdmx.oecd.org/public/rest"

# ----------------------------- Core helpers -------------------------------- #

def probe_structure(
    agency: str,
    flow: str,
    ver: str,
    base_url: str = BASE,
    timeout: int = 60,
    max_attempts: int = 6,
    max_backoff: int = 120,
    verbose: bool = False,
) -> Tuple[List[str], Dict[str, List[Dict[str, str]]]]:
    """
    Get series dimension order and codes with retries on 429/transient errors.
    Honors Retry-After header (seconds or HTTP-date).
    """
    url = f"{base_url}/data/{agency},{flow},{ver}"
    attempt = 0
    while attempt < max_attempts:
        attempt += 1
        try:
            if verbose:
                print(f"[probe] attempt {attempt}/{max_attempts} -> {url}")
            r = requests.get(
                url,
                params={"detail": "serieskeysonly", "firstNObservations": "0", "format": "sdmx-json"},
                headers={
                    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
                    "Accept": "application/json,text/csv,*/*;q=0.8",
                },
                timeout=timeout,
            )
            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")
                wait = None
                if retry_after:
                    try:
                        wait = int(retry_after)
                    except Exception:
                        try:
                            dt = parsedate_to_datetime(retry_after)
                            wait = max(0, (dt - datetime.datetime.now(datetime.timezone.utc)).total_seconds())
                        except Exception:
                            wait = None
                if wait is None:
                    wait = min(max_backoff, 2 ** attempt + random.random() * 4)
                if verbose:
                    print(f"[probe] 429 -> sleeping {wait:.1f}s")
                time.sleep(wait)
                continue
            r.raise_for_status()
            js = r.json()
            series_dims = js["structure"]["dimensions"]["series"]
            dim_order = [d["id"] for d in series_dims]
            codes = {
                d["id"]: [{"id": v["id"], "name": v.get("name", v.get("id"))} for v in d.get("values", [])]
                for d in series_dims
            }
            return dim_order, codes
        except requests.RequestException as exc:
            if attempt >= max_attempts:
                raise
            wait = min(max_backoff, 2 ** attempt + random.random() * 4)
            if verbose:
                print(f"[probe] request error: {exc}; retrying in {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError("Failed to probe SDMX structure after retries")


def _pick_code_from(
    values: List[Mapping[str, str]],
    prefer_ids: Iterable[str] = (),
    prefer_name_contains: Iterable[str] = (),
) -> str:
    """Pick a single code from a values list by id or name contains; fallback to first."""
    # try explicit ids
    prefer_ids = [p.upper() for p in prefer_ids]
    for pid in prefer_ids:
        for v in values:
            if v["id"].upper() == pid:
                return v["id"]
    # try substring on label/name
    for substr in prefer_name_contains:
        s = substr.lower()
        for v in values:
            if s in str(v.get("name", "")).lower():
                return v["id"]
    # fallback: first available
    return values[0]["id"] if values else ""


def _normalize_selector(
    dim_id: str,
    values: List[Mapping[str, str]],
    selector: Union[str, Iterable[str], Mapping[str, Iterable[str]]]
) -> str:
    """
    Return a valid SDMX code expression for this dimension:
      - str -> returned as-is
      - iterable[str] -> joined with '+'
      - dict with keys 'prefer_ids' and/or 'prefer_name_contains' -> picked by _pick_code_from
    """
    if isinstance(selector, str):
        return selector  # may be "", "TL2+CTRY", "ISCED11_5T8", etc.
    if isinstance(selector, Mapping):
        return _pick_code_from(
            values,
            selector.get("prefer_ids", ()),
            selector.get("prefer_name_contains", ()),
        )
    # assume iterable of codes (multi-select)
    try:
        codes = list(selector)
        return "+".join(codes)
    except TypeError:
        raise TypeError(f"Unsupported selector type for dimension {dim_id}: {type(selector)}")


def build_key(
    dim_order: List[str],
    codes_by_dim: Dict[str, List[Mapping[str, str]]],
    selectors: Mapping[str, Union[str, Iterable[str], Mapping[str, Iterable[str]]]],
    allow_missing: bool = True
) -> str:
    """
    Build a full SDMX key string (dot-separated) in the dataset's dimension order.
    `selectors` maps dimension -> selector (string, list of strings, or preference dict).
    If a dimension isn't specified:
      - if allow_missing=True, pick a reasonable default (first available, or empty "")
      - else raise KeyError
    """
    parts = []
    for dim in dim_order:
        if dim in selectors:
            expr = _normalize_selector(dim, codes_by_dim.get(dim, []), selectors[dim])
        else:
            if not allow_missing:
                raise KeyError(f"Selector missing for dimension: {dim}")
            # empty string often means 'all' for OECD CFE EDS in some dims (e.g., REF_AREA)
            vals = codes_by_dim.get(dim, [])
            expr = vals[0]["id"] if vals else ""
        parts.append(expr)
    return ".".join(parts)


def fetch_sdmx_csv(
    agency: str,
    flow: str,
    ver: str,
    selectors: Mapping[str, Union[str, Iterable[str], Mapping[str, Iterable[str]]]],
    *,
    base_url: str = BASE,
    start: Optional[str] = None,
    end: Optional[str] = None,
    dimension_at_obs: str = "AllDimensions",
    timeout: int = 60,
    verbose: bool = False,
    max_attempts: int = 8,
    max_backoff: int = 300,
    fixed_delay_before_first_attempt: Optional[float] = None,
) -> Tuple[pd.DataFrame, Dict[str, List[Mapping[str, str]]], List[str], str]:
    """
    Probe structure, build key and fetch CSV. Retries 429 and transient errors with
    exponential backoff + jitter. Honors Retry-After header (seconds or HTTP date).
    """
    dim_order, codes = probe_structure(agency, flow, ver, base_url=base_url, timeout=timeout)
    key = build_key(dim_order, codes, selectors)

    params = {"format": "csvfilewithlabels", "dimensionAtObservation": dimension_at_obs}
    if start:
        params["startPeriod"] = start
    if end:
        params["endPeriod"] = end

    url = f"{base_url}/data/{agency},{flow},{ver}/{key}"

    if fixed_delay_before_first_attempt:
        if verbose:
            print(f"[fetch] fixed sleep for {fixed_delay_before_first_attempt:.1f}s before first attempt")
        time.sleep(fixed_delay_before_first_attempt)

    attempt = 0
    while attempt < max_attempts:
        attempt += 1
        try:
            if verbose:
                print(f"[fetch] attempt {attempt}/{max_attempts} -> {url} params={params}")
            r = requests.get(
                url,
                params=params,
                headers={
                    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"),
                    "Accept": "application/json,text/csv,*/*;q=0.8",
                },
                timeout=timeout,
            )

            # Handle rate limit
            if r.status_code == 429:
                retry_after = r.headers.get("Retry-After")
                wait = None
                if retry_after:
                    # try seconds
                    try:
                        wait = int(retry_after)
                    except Exception:
                        # try HTTP-date
                        try:
                            dt = parsedate_to_datetime(retry_after)
                            wait = max(0, (dt - datetime.datetime.now(datetime.timezone.utc)).total_seconds())
                        except Exception:
                            wait = None
                if wait is None:
                    # exponential backoff with jitter
                    wait = min(max_backoff, 2 ** attempt + random.random() * 4)
                if verbose:
                    print(f"[fetch] 429 Too Many Requests -> sleeping {wait:.1f}s (attempt {attempt})")
                time.sleep(wait)
                continue

            # Raise for other HTTP errors
            r.raise_for_status()
            # success
            break

        except requests.RequestException as exc:
            if attempt >= max_attempts:
                if verbose:
                    print(f"[fetch] final failure after {attempt} attempts: {exc}")
                raise
            # exponential backoff with jitter
            wait = min(max_backoff, 2 ** attempt + random.random() * 4)
            if verbose:
                print(f"[fetch] request error: {exc}; retrying in {wait:.1f}s (attempt {attempt})")
            time.sleep(wait)
    else:
        raise RuntimeError("Failed to fetch SDMX CSV after retries")

    df = pd.read_csv(io.StringIO(r.text))
    if verbose:
        print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} cols  Key: {key}")
    return df, codes, dim_order, key

# ----------------------------- Ready-made examples -------------------------- #

def load_education_attainment_tertiary(
    start="2010", end="2024", verbose=False
) -> pd.DataFrame:
    """
    OECD CFE EDS — Education attainment (tertiary, ages 25–64), TL2 + countries, annual, % of pop.
    """
    agency = "OECD.CFE.EDS"
    flow = "DSD_REG_EDU@DF_ATTAIN"
    ver = "2.0"

    selectors = {
        # Annual
        "FREQ": {"prefer_ids": ("A",)},
        # Both TL2 regions and countries
        "TERRITORIAL_LEVEL": "TL2+CTRY",
        # All areas (leave empty in this flow to mean 'all')
        "REF_AREA": "",
        # Not applicable / default territorial type
        "TERRITORIAL_TYPE": {"prefer_ids": ("_Z",)},
        # Education measure: education attainment share
        "MEASURE": {"prefer_ids": ("NEAC_SHARE_EA",)},
        # Age 25–64
        "AGE": {"prefer_ids": ("Y25T64",)},
        # Sex total
        "SEX": {"prefer_ids": ("_T", "T"), "prefer_name_contains": ("total",)},
        # Tertiary (ISCED 5–8)
        "EDUCATION_LEV": {"prefer_ids": ("ISCED11_5T8",)},
        # Mean
        "STATISTICAL_OPERATION": {"prefer_ids": ("MEAN",)},
        # Percentage in population by sex-age
        "UNIT_MEASURE": {"prefer_ids": ("PT_POP_SEX_AGE",)},
    }

    df, *_ = fetch_sdmx_csv(
        agency, flow, ver, selectors, start=start, end=end, verbose=verbose
    )
    return df

def load_enrolment_rate(
    *,
    ref_areas,                         # list like ["AUT","AUS","AU1",...]
    age: str = "Y15T19",
    sexes=("F","M","_T"),
    territorial_level: str = "CTRY+TL2",
    start="2013",
    end=None,
    verbose=False
):
    """
    OECD CFE EDS — Education Enrolment Rate (DF_EDU).
    Example measure: ENRL_RATE for ages 15–19, by TL2 + Country.
    """
    agency = "OECD.CFE.EDS"
    flow   = "DSD_REG_EDU@DF_EDU"
    ver    = "2.0"

    selectors = {
        "FREQ": {"prefer_ids": ("A",)},              # Annual
        "TERRITORIAL_LEVEL": territorial_level,      # "CTRY+TL2"
        "REF_AREA": ref_areas,                       # list -> "AUT+AUS+AU1+..."
        "TERRITORIAL_TYPE": "",                      # empty is common here
        "MEASURE": "ENRL_RATE",
        "AGE": age,                                  # "Y15T19"
        "SEX": list(sexes),                          # e.g. ("F","M","_T")
        "EDUCATION_LEV": "_T",                       # total level
        # Many DF_EDU series either omit these or default; let the picker choose.
        "STATISTICAL_OPERATION": {"prefer_ids": ("MEAN",)},
        "UNIT_MEASURE": {"prefer_ids": ("PT_POP_SEX_AGE","PT","RATE")},
    }

    df, *_ = fetch_sdmx_csv(
        agency, flow, ver, selectors, start=start, end=end, verbose=verbose
    )
    return df

def load_health(
    measure_code: str,
    *,
    ages: Union[str, Iterable[str]] = "Y0",
    sexes: Union[str, Iterable[str]] = "",
    start="2010",
    end="2024",
    verbose=False,
    fixed_delay_before_first_attempt: Optional[float] = None,
) -> pd.DataFrame:
    agency = "OECD.CFE.EDS"
    flow = "DSD_REG_HEALTH@DF_HEALTH"
    ver = "2.0"

    selectors = {
        "FREQ": {"prefer_ids": ("A",)},
        "TERRITORIAL_LEVEL": "TL2+CTRY",
        "REF_AREA": "",
        "TERRITORIAL_TYPE": "",
        "MEASURE": measure_code,
        "AGE": ages,
        "SEX": sexes,
        "UNIT_MEASURE": "",
    }

    df, *_ = fetch_sdmx_csv(
        agency, flow, ver, selectors,
        start=start, end=end, verbose=verbose,
        fixed_delay_before_first_attempt=fixed_delay_before_first_attempt
    )
    return df



In [140]:
if __name__ == "__main__":
    # 1) Education: tertiary attainment, 25–64
    #edu = load_education_attainment_tertiary(start="2010", end="2024", verbose=True)
    #print("Education sample:")
    # print(edu.head())

    # 2) Health: life expectancy at birth (LFEXP), total sex, age Y0
    #lifeexp = load_health("LFEXP", ages="Y0", sexes="_T", start="2010", end="2024", verbose=True)
    #print("Life expectancy sample:")
    #print(lifeexp.head())

    # 3) Health: infant mortality, multiple age buckets combined
    mort_inf = load_health("MORT_INFANT", ages=["Y_GE15", "Y_LT15", "Y_LT1", "Y0"], sexes="_T",
                           start="2010", end="2024", verbose=True)
    print("Infant mortality sample:")
    print(mort_inf.head())

    # 4) Education: enrolment rate for ages 15–19,
    enrol_15_19 = load_enrolment_rate(
    ref_areas="",
    age="Y15T19",
    sexes=("F","M","_T"),
    start="2010",
    end=2024,         # or "2024"
    verbose=True)
    print("Enrolment rate sample:")
    print(enrol_15_19.head())




[fetch] attempt 1/8 -> https://sdmx.oecd.org/public/rest/data/OECD.CFE.EDS,DSD_REG_HEALTH@DF_HEALTH,2.0/A.TL2+CTRY...MORT_INFANT.Y_GE15+Y_LT15+Y_LT1+Y0._T. params={'format': 'csvfilewithlabels', 'dimensionAtObservation': 'AllDimensions', 'startPeriod': '2010', 'endPeriod': '2024'}
Loaded 7,732 rows × 32 cols  Key: A.TL2+CTRY...MORT_INFANT.Y_GE15+Y_LT15+Y_LT1+Y0._T.
Infant mortality sample:
  STRUCTURE                                STRUCTURE_ID  \
0  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
1  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
2  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
3  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   
4  DATAFLOW  OECD.CFE.EDS:DSD_REG_HEALTH@DF_HEALTH(2.0)   

                STRUCTURE_NAME ACTION FREQ Frequency of observation  \
0  Health statistics - Regions      I    A                   Annual   
1  Health statistics - Regions      I    A                   Annual   
2  Health statistics - Regions      I  

In [148]:
print(enrol_15_19[enrol_15_19['Country'] == "Colombia"][['TIME_PERIOD', 'OBS_VALUE']])

       TIME_PERIOD  OBS_VALUE
1089          2015        0.0
1090          2014        0.0
1091          2014        0.0
1092          2022       62.8
1093          2022       62.2
...            ...        ...
14598         2014        0.0
14599         2012        0.0
14600         2019        0.0
14601         2012        0.0
14602         2012        0.0

[1326 rows x 2 columns]


In [109]:
vars = enrol_15_19[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
data_2022 = vars[vars['TIME_PERIOD'] == 2022]
data_2020 = vars[vars['TIME_PERIOD'] == 2020]
data_2022_us = vars[vars['COUNTRY'] == 'USA']
# Separate the two territorial levels
enrollment_country_data = vars[vars['Territorial level'] == 'Country']
enrollment_regional_data = vars[vars['Territorial level'] == 'TL2']

# Make sure TIME_PERIOD is numeric
enrollment_country_data['TIME_PERIOD'] = pd.to_numeric(enrollment_country_data['TIME_PERIOD'], errors='coerce')

# Sort by country and year (descending)
sorted_df = enrol_15_19.sort_values(['REF_AREA', 'TIME_PERIOD', 'Sex'], ascending=[True, False, False])

# Keep only the most recent entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='first')

circa_2024 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
enrollment_country_data = circa_2024[circa_2024['Territorial level'] == 'Country']
enrollment_regional_data = circa_2024[circa_2024['Territorial level'] == 'TL2']

# Keep only the OLDEST entry for each country
latest_country_data = sorted_df.drop_duplicates(subset=['REF_AREA', 'Sex'], keep='last')

circa_2010 = latest_country_data[['STRUCTURE_NAME', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'MEASURE', 'Measure',  'Age',  'Sex',  'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country']]
enrollment_country_data_2010 = circa_2010[circa_2010['Territorial level'] == 'Country']
enrollment_regional_data_2010 = circa_2010[circa_2010['Territorial level'] == 'TL2']


C:\Users\lopez\AppData\Local\Temp\ipykernel_32144\1333821890.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enrollment_country_data['TIME_PERIOD'] = pd.to_numeric(enrollment_country_data['TIME_PERIOD'], errors='coerce')


In [84]:
enrollment_country_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_countries_complete.csv", index=False)
enrollment_regional_data.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\enrollment_regional_data.csv", index=False)

In [56]:
# make child mortality child survival rate
mort_inf['survival_rate'] = (1- (mort_inf['OBS_VALUE'] / 1000))*100

mort_inf.drop(columns=['OBS_VALUE'], inplace=True)

mort_inf['OBS_VALUE'] = mort_inf['survival_rate']


In [57]:
indicators = {}

# education (prefer TL2 regional frame if present)
if 'EDU_attainment_regional_data' in globals():
    indicators['edu'] = EDU_attainment_regional_data.copy()
elif 'edu' in globals():
    indicators['edu'] = edu.copy()
elif 'education_df' in globals():
    indicators['edu'] = education_df[education_df.get('Territorial level','') == 'TL2'].copy()
else:
    print("Warning: no education (TL2) dataframe found; 'edu' not added to indicators")

# enrolment (DF_EDU / enrol_15_19)
if 'EDU_enrollment_regional_data' in globals():
    indicators['enrol'] = EDU_enrollment_regional_data.copy()
elif 'enrol_15_19' in globals():
    indicators['enrol'] = enrol_15_19[enrol_15_19.get('Territorial level','') == 'TL2'].copy()

# health (life expectancy / child mortality) — prefer processed TL2 frame
if 'Health_life_expectancy_regional_data' in globals():
    indicators['health'] = Health_life_expectancy_regional_data.copy()
elif 'lifeexp' in globals():
    indicators['health'] = lifeexp[lifeexp.get('Territorial level','') == 'TL2'].copy()

# health (life expectancy / child mortality) — prefer processed TL2 frame
if 'Health_child_mortality_regional_data' in globals():
    indicators['child_mortality'] = Health_child_mortality_regional_data.copy()
elif 'mort_inf' in globals():
    indicators['child_mortality'] = mort_inf[mort_inf.get('Territorial level','') == 'TL2'].copy()

# quick report
for k, v in indicators.items():
    print(f"indicator '{k}': {len(v):,} rows — columns: {list(v.columns)[:6]}")

indicator 'edu': 6,771 rows — columns: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation']
indicator 'enrol': 13,813 rows — columns: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation']
indicator 'health': 6,035 rows — columns: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation']
indicator 'child_mortality': 7,135 rows — columns: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation']


In [132]:
def _norm_clip(series, lo=0.0, hi=1.0):
    s = pd.to_numeric(series, errors="coerce")
    return s.clip(lower=lo, upper=hi)

def _life_index(le_years: pd.Series, lo=20.0, hi=85.0):
    return _norm_clip((le_years - lo) / (hi - lo), 0.0, 1.0)

def _pct_index(pct: pd.Series):
    return _norm_clip(pct / 100.0, 0.0, 1.0)

def _gmean(a: pd.Series, b: pd.Series, c: pd.Series, d: pd.Series):
    return (a + b + c + d) * 0.25

def build_national_hci(Life: pd.DataFrame,
                       survival: pd.DataFrame,
                       attainment: pd.DataFrame,
                       enrollment: pd.DataFrame,
                       area_col: str = "Reference area",
                       value_col: str = "OBS_VALUE"):
    
    L = Life[[area_col, value_col]].rename(columns={value_col: "LE"})
    S = survival[[area_col, value_col]].rename(columns={value_col: "Survival"})
    A = attainment[[area_col, value_col]].rename(columns={value_col: "Attainment"})
    E = enrollment[[area_col, value_col]].rename(columns={value_col: "Enrollment"})
    df = L.merge(S, on=area_col, how="inner").merge(A, on=area_col, how="inner").merge(E, on=area_col, how="inner")
    df["LE_idx"] = _life_index(df["LE"])
    df["Survival_idx"] = _pct_index(df["Survival"])
   
    df["Attain_idx"] = _pct_index(df["Attainment"])
    df["Enroll_idx"] = _pct_index(df["Enrollment"])
   
    df["HCI_composite"] = _gmean(df["LE_idx"], df["Survival_idx"], df["Attain_idx"], df["Enroll_idx"])
    cols = [area_col, "LE", "Survival", "Attainment", "Enrollment",
            "LE_idx", "Survival_idx",  "Attain_idx", "Enroll_idx", "HCI_composite"]
    return df[cols].sort_values("HCI_composite", ascending=False).reset_index(drop=True)

In [ ]:
# make child mortality child survival rate
Health_mortality_country_data_2010['survival_rate'] = (1- (Health_mortality_country_data_2010['OBS_VALUE'] / 1000))*100
Health_mortality_country_data_2010.drop(columns=['OBS_VALUE'], inplace=True)
Health_mortality_country_data_2010['OBS_VALUE'] = Health_mortality_country_data_2010['survival_rate']

Health_mortality_country_data['survival_rate'] = (1- (Health_mortality_country_data['OBS_VALUE'] / 1000))*100
Health_mortality_country_data.drop(columns=['OBS_VALUE'], inplace=True)
Health_mortality_country_data['OBS_VALUE'] = Health_mortality_country_data['survival_rate']

enrollment_country_data_2010["circa"] = 2010
enrollment_country_data["circa"] = 2024

EDU_country_data_2010["circa"] = 2010
EDU_country_data_with_additions["circa"] = 2024

Health_country_data_2010["circa"] = 2010
Health_country_data["circa"] = 2024

Health_mortality_country_data_2010["circa"] = 2010
Health_mortality_country_data["circa"] = 2024

enrollment_country_data_2010[enrollment_country_data_2010["Sex"] == "Total"],
enrollment_country_data_2010.loc[enrollment_country_data_2010['Country'] == 'Colombia', 'OBS_VALUE'] = 60
enrollment_country_data[enrollment_country_data["Sex"] == "Total"],


EDU_country_data_2010[EDU_country_data_2010["Sex"] == "Total"],
EDU_country_data_with_additions[EDU_country_data_with_additions["Sex"] == "Total"],
Health_country_data_2010[Health_country_data_2010["Sex"] == "Total"],
Health_country_data[Health_country_data["Sex"] == "Total"],
Health_mortality_country_data_2010[Health_mortality_country_data_2010["Sex"] == "Total"],
Health_mortality_country_data[Health_mortality_country_data["Sex"] == "Total"]

In [167]:
country_year_hci_2010 = build_national_hci(
     Life=Health_country_data_2010, 
     survival=Health_mortality_country_data_2010,
     attainment=EDU_country_data_2010,
     enrollment=enrollment_country_data_2010,
     area_col = "Reference area",
     value_col= "OBS_VALUE"
)

country_year_hci_2024 = build_national_hci(
     Life=Health_country_data, 
     survival=Health_mortality_country_data,
     attainment=EDU_country_data,
     enrollment=enrollment_country_data,
     area_col = "Reference area",
     value_col= "OBS_VALUE"
)

In [168]:
country_year_hci_2024

,Reference area,LE,Survival,Attainment,Enrollment,LE_idx,Survival_idx,Attain_idx,Enroll_idx,HCI_composite
0,Ireland,84.6,90.909100,56.6,94.8,0.993846,0.909091,0.566,0.948,0.854234
1,Ireland,84.6,90.909100,56.6,94.8,0.993846,0.909091,0.566,0.948,0.854234
2,Ireland,84.6,90.909100,56.6,94.8,0.993846,0.909091,0.566,0.948,0.854234
3,Ireland,84.6,90.909100,56.6,93.9,0.993846,0.909091,0.566,0.939,0.851984
4,Ireland,84.6,90.909100,56.6,93.9,0.993846,0.909091,0.566,0.939,0.851984
...,...,...,...,...,...,...,...,...,...,...
844,Croatia,81.8,90.909100,30.4,39.3,0.950769,0.909091,0.304,0.393,0.639215
845,Brazil,72.5,90.909099,14.8,69.0,0.807692,0.909091,0.148,0.690,0.638696
846,Croatia,75.5,90.909100,30.4,48.5,0.853846,0.909091,0.304,0.485,0.637984
847,Croatia,78.6,90.909100,30.4,39.3,0.901538,0.909091,0.304,0.393,0.626907


In [169]:
country_year_hci_2024["CIRCA"] = 2024
country_year_hci_2010["CIRCA"] = 2010
HCI_2010_2024 = pd.concat([country_year_hci_2010, country_year_hci_2024], ignore_index=True)

In [170]:
HCI_2010_2024

,Reference area,LE,Survival,Attainment,Enrollment,LE_idx,Survival_idx,Attain_idx,Enroll_idx,HCI_composite,CIRCA
0,Canada,83.54,90.909100,50.0,86.8,0.977538,0.909091,0.500,0.868,0.813657,2010
1,Canada,83.54,90.909099,50.0,86.8,0.977538,0.909091,0.500,0.868,0.813657,2010
2,Canada,83.54,90.909099,50.0,86.8,0.977538,0.909091,0.500,0.868,0.813657,2010
3,Norway,83.30,90.909100,36.9,100.0,0.973846,0.909091,0.369,1.000,0.812984,2010
4,Norway,83.30,90.909100,36.9,100.0,0.973846,0.909091,0.369,1.000,0.812984,2010
...,...,...,...,...,...,...,...,...,...,...,...
1693,Croatia,81.80,90.909100,30.4,39.3,0.950769,0.909091,0.304,0.393,0.639215,2024
1694,Brazil,72.50,90.909099,14.8,69.0,0.807692,0.909091,0.148,0.690,0.638696,2024
1695,Croatia,75.50,90.909100,30.4,48.5,0.853846,0.909091,0.304,0.485,0.637984,2024
1696,Croatia,78.60,90.909100,30.4,39.3,0.901538,0.909091,0.304,0.393,0.626907,2024


In [171]:
HCI_2010_2024.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\HCI_2010_2024.csv", index=False)

In [130]:
dfs = [
    enrollment_country_data_2010[enrollment_country_data_2010["Sex"] == "Total"],
    enrollment_country_data[enrollment_country_data["Sex"] == "Total"],
    EDU_country_data_2010[EDU_country_data_2010["Sex"] == "Total"],
    EDU_country_data_with_additions[EDU_country_data_with_additions["Sex"] == "Total"],
    Health_country_data_2010[Health_country_data_2010["Sex"] == "Total"],
    Health_country_data[Health_country_data["Sex"] == "Total"],
    Health_mortality_country_data_2010[Health_mortality_country_data_2010["Sex"] == "Total"],
    Health_mortality_country_data[Health_mortality_country_data["Sex"] == "Total"]
]

# Concatenate all filtered DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

print(combined_df["Country"].unique())

['Australia' 'Austria' 'Belgium' 'Bulgaria' 'Brazil' 'Canada'
 'Switzerland' 'Chile' 'Colombia' 'Costa Rica' 'Cyprus' 'Czechia'
 'Germany' 'Denmark' 'Spain' 'Estonia'
 'European Union (27 countries from 01/02/2020)'
 'European Union (28 countries)' 'Finland' 'France' 'United Kingdom'
 'Greece' 'Croatia' 'Hungary' 'Ireland' 'Iceland' 'Israel' 'Italy' 'Japan'
 'Korea' 'Lithuania' 'Luxembourg' 'Latvia' 'Mexico' 'North Macedonia'
 'Malta' 'Netherlands' 'Norway' 'New Zealand' 'Poland' 'Portugal'
 'Romania' 'Russia' 'Serbia' 'Slovak Republic' 'Slovenia' 'Sweden'
 'Türkiye' 'United States' 'Euro area (20 countries)'
 'Euro area (15 countries)' 'Montenegro' 'Argentina'
 'China (People’s Republic of)' 'Indonesia' 'India' 'Peru' 'South Africa']


In [131]:
combined_df.to_csv("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\data\\combined_indicators_20102024.csv", index=False)

In [53]:
regional_with_gap = {}

for name, df in indicators.items():
    # 1) TL2-only
    reg = df[df["Territorial level"] == "TL2"].copy()

    # 2) Ensure a 'Country' column to group by
    if "Country" not in reg.columns:
        # Try common alternatives from OECD CSVs:
        for candidate in ["Reference country", "REF_AREA_PARENT", "REF_AREA_COUNTRY"]:
            if candidate in reg.columns:
                reg["Country"] = reg[candidate]
                break
        else:
            # Fallback: derive from REF_AREA (adjust to your coding scheme)
            # Many TL2 codes start with the ISO3 country, or at least 3-letter country.
            reg["Country"] = reg["REF_AREA"].str[:3]

    # 3) Gap per (year, country): max - min across TL2 regions
    gap_by_year_country = (
        reg.groupby(["TIME_PERIOD", "Country"], as_index=False)["OBS_VALUE"]
           .agg(gap_obs_value=lambda x: x.max() - x.min())
    )

    # 4) Attach the gap to every regional row
    reg = reg.merge(gap_by_year_country, on=["TIME_PERIOD", "Country"], how="left")

    # 5) Store result
    regional_with_gap[name] = reg

In [33]:
all_gaps = {}

for name, reg in regional_with_gap.items():
    # Group by country + year
    gap_table = (
        reg.groupby(["Country", "TIME_PERIOD"])
        .agg(
            gap_obs_value=("gap_obs_value", "first"),  # same per country-year
            n_regions=("REF_AREA", "nunique")          # number of TL2 regions in that year
        )
        .reset_index()
        .sort_values(["Country", "TIME_PERIOD"])
    )

    # Find countries with >5 unique TL2 regions across the whole dataset
    region_counts = (
        reg.groupby("Country")["REF_AREA"]
        .nunique()
    )
    countries_with_5plus = region_counts[region_counts > 5].index

    # Filter gap_table to only those countries
    gap_table = gap_table[gap_table["Country"].isin(countries_with_5plus)]

    all_gaps[name] = gap_table

# Example: see for education
print("Education gaps (countries with >5 TL2 regions):")
print(all_gaps["edu"].head())


Education gaps (countries with >5 TL2 regions):
     Country  TIME_PERIOD  gap_obs_value  n_regions
0  Australia         2016           28.9          8
1  Australia         2017           26.5          8
2  Australia         2018           31.5          8
3  Australia         2019           25.2          8
4  Australia         2020           24.0          8


In [91]:
selected_countries = [
    "United States", "Czechia",  "Canada", "Spain",
    "France", "United Kingdom", "Colombia", "Romania",  
    "Greece", "Bulgaria", "Mexico", "Chile"
    
]

filtered_edu = all_gaps["edu"][all_gaps["edu"]["Country"].isin(selected_countries)]



In [77]:
df = filtered_edu  # columns: Country, TIME_PERIOD, gap_obs_value, n_regions

# add a per-country sort key (mean gap)
base = (
    alt.Chart(df)
    .transform_joinaggregate(sort_key='mean(gap_obs_value)', groupby=['Country'])
)

line = (
    base.mark_line(color="#9C0505")
    .encode(
        x=alt.X('TIME_PERIOD:O', title='Year', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('gap_obs_value:Q', title='Gap (pp, max–min of TL2)'),
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions']
    )
)

points = (
    base.mark_point(color='red', filled=True, size=50)
    .encode(
        x='TIME_PERIOD:O',
        y='gap_obs_value:Q'
    )
)

panel = (line + points).properties(width=160, height=120)

panel = (
    base.mark_line(color='#9C0505', point=True)
    .encode(
        x=alt.X('TIME_PERIOD:O', title='Year', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('gap_obs_value:Q', title='Gap (pp, max–min of TL2)'),
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions']
    )
    .properties(width=160, height=120)
)

chart = panel.facet(
    facet=alt.Facet(
        'Country:N',
        sort=alt.SortField(field='sort_key', order='descending'),  # << fix
        title=None
    ),
    columns=6
).resolve_scale(y='shared').properties(
    title='Education Attainment Gap (Tertiary, 25–64) — Selected Countries (2010–2023)'
).configure_axis(grid=True)

chart


NameError: name 'filtered_edu' is not defined

In [76]:
# ...existing code...
df = filtered_edu  # columns: Country, TIME_PERIOD, gap_obs_value, n_regions

# compute first / middle / last year values (fall back to whatever exists)
_years = sorted(df['TIME_PERIOD'].dropna().unique())
if len(_years) >= 3:
    axis_years = [int(_years[0]), int(_years[len(_years)//2]), int(_years[-1])]
else:
    axis_years = [int(y) for y in _years]

x_axis = alt.X('TIME_PERIOD:O', title='Year', axis=alt.Axis(values=axis_years, labelAngle=0))

# add a per-country sort key (mean gap)
base = (
    alt.Chart(df)
    .transform_joinaggregate(sort_key='mean(gap_obs_value)', groupby=['Country'])
)

line = (
    base.mark_line()
    .encode(
        x=x_axis,
        y=alt.Y('gap_obs_value:Q', title='Gap (pp, max–min of TL2)'),
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions'],
        color=alt.value("#9C0505")
    )
)

points = (
    base.mark_point(filled=True, size=50)
    .encode(
        x=x_axis,
        y='gap_obs_value:Q',
        tooltip=['Country', 'TIME_PERIOD', alt.Tooltip('gap_obs_value:Q', title='Gap'), 'n_regions'],
        color=alt.value("#9C0505")
    )
)

panel = (line + points).properties(width=160, height=120)

chart = panel.facet(
    facet=alt.Facet(
        'Country:N',
        sort=alt.EncodingSortField(field='sort_key', order='descending'),
        title=None
    ),
    columns=6
).resolve_scale(y='shared').properties(
    title='Education Attainment Gap (Tertiary, 25–64) — Selected Countries (2010–2023)'
).configure_axis(grid=True)

chart

chart.save("C:\\Users\\lopez\\github\\capp30239\\static_visualization_project\\graphs\\trend_education_attainment_gap_selected_countries.png")

NameError: name 'filtered_edu' is not defined